# Aula 4 — Performance e funcionamento interno do Spark
### Do "funciona" ao "funciona bem"

Na aula anterior vimos a API. Com 15 linhas, qualquer ferramenta é rápida.
Hoje usamos **milhões de linhas** para entender o que o Spark faz por baixo e como tomar decisões de performance.

**Objetivos**
- Entender job, stage e task, e o que é um shuffle
- Ler um plano de execução com `explain()`
- Escolher a estratégia de join correta (broadcast vs sort-merge)
- Detectar e corrigir *data skew*
- Saber o que o AQE faz automaticamente
- Decidir quando usar cache e quando evitar UDFs

---
## 1. Setup

In [1]:
# !pip install -q pyspark==4.0.1 pyarrow

In [2]:
import io, re, time, contextlib
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Aula 4 - Performance")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "false")   # desligado de propósito; ligamos na seção 8
    .config("spark.ui.showConsoleProgress", "false") # esconde as barras de progresso
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Núcleos :", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 20:17:19 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/10/28 20:17:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 20:17:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark   : 4.0.1
Núcleos : 24
Spark UI: http://10.23.20.50:4040


Funções auxiliares usadas no notebook:

- `timer` mede o tempo e coloca um nome no job (aparece na Spark UI).
- `run` força a execução **sem trazer dados para o driver**, usando o formato `noop`. É o jeito correto de medir performance: `show()` lê só algumas linhas e `collect()` mede também a transferência para o Python.
- `count_shuffles` conta quantos `Exchange` (shuffles) existem no plano.

In [3]:
@contextlib.contextmanager
def timer(label):
    spark.sparkContext.setJobDescription(label)
    t0 = time.perf_counter()
    yield # quando finaliza
    print(f"{label:<45} {time.perf_counter() - t0:6.2f} s")
    spark.sparkContext.setJobDescription(None)

def run(df):
    df.write.format("noop").mode("overwrite").save()

def plan_text(df, mode="simple"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

def count_shuffles(df):
    return len(re.findall(r"\bExchange\b", plan_text(df)))

def partition_balance(df, *keys, n=16, label=""):
    """Redistribui por chave e mostra o tamanho da maior partição vs a mediana."""
    s = (df.repartition(n, *keys)
           .groupBy(F.spark_partition_id()).count()
           .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
           .first())
    print(f"{label:<12} maior = {s['max']:>9,} | mediana = {s['median']:>9,} | razão = {s['max'] / s['median']:.1f}x")

---
## 2. Dataset grande

Vamos gerar três tabelas no estilo da aula 3, mas em escala:

| Tabela | Linhas | Papel |
|--------|--------|-------|
| `orders` | 5 milhões | Fatos (pedidos) |
| `customers` | 200 mil | Dimensão média |
| `products` | 1 mil | Dimensão pequena |

`spark.range()` gera os dados já distribuídos, sem passar pelo Python. Salvamos em Parquet para que as leituras seguintes sejam I/O real.

> Se a sua máquina for modesta (ou no Colab gratuito), reduza `N_ORDERS` para 2 milhões.

In [4]:
N_ORDERS    = 5_000_000
N_CUSTOMERS = 200_000
N_PRODUCTS  = 1_000
REGIONS     = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES  = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
BASE        = "/tmp/aula4"

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

orders_gen = (
    spark.range(N_ORDERS).withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.rand(1) * N_CUSTOMERS).cast("int"))
    .withColumn("product_id",  (F.rand(2) * N_PRODUCTS).cast("int"))
    .withColumn("quantity",    (F.rand(3) * 5 + 1).cast("int"))
    .withColumn("price",       F.round(F.rand(4) * 490 + 10, 2))
    .withColumn("region",      pick(REGIONS, 5))
    .withColumn("order_date",  F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand(6) * 365).cast("int")))
)

customers_gen = (
    spark.range(N_CUSTOMERS).withColumnRenamed("id", "customer_id")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("segment", pick(["Consumer", "Corporate", "Small Business"], 11))
    .withColumn("signup_year", (F.rand(12) * 6 + 2019).cast("int"))
)

products_gen = (
    spark.range(N_PRODUCTS).withColumnRenamed("id", "product_id")
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("product_id")))
    .withColumn("category", pick(CATEGORIES, 21))
)

with timer("Gerar e salvar as 3 tabelas"):
    orders_gen.write.mode("overwrite").parquet(f"{BASE}/orders")
    customers_gen.write.mode("overwrite").parquet(f"{BASE}/customers")
    products_gen.write.mode("overwrite").parquet(f"{BASE}/products")

orders    = spark.read.parquet(f"{BASE}/orders")
customers = spark.read.parquet(f"{BASE}/customers")
products  = spark.read.parquet(f"{BASE}/products")

orders.show(5)

Gerar e salvar as 3 tabelas                     2.96 s
+--------+-----------+----------+--------+------+---------+----------+
|order_id|customer_id|product_id|quantity| price|   region|order_date|
+--------+-----------+----------+--------+------+---------+----------+
| 2291666|      73092|       417|       1| 98.89|  Midwest|2024-12-19|
| 2291667|     186251|       989|       3|317.22|    North|2024-01-28|
| 2291668|     140692|       483|       4|183.57|    North|2024-01-20|
| 2291669|      32718|       607|       5| 113.4|    North|2024-07-17|
| 2291670|       1265|       401|       5|411.26|Northeast|2024-04-30|
+--------+-----------+----------+--------+------+---------+----------+
only showing top 5 rows


### 2.1 Pandas vs Spark com 5 milhões de linhas

Mesma consulta nas duas ferramentas: receita total por região.
O resultado costuma surpreender.

In [5]:
with timer("Pandas: ler Parquet + groupby"):
    pdf = pd.read_parquet(f"{BASE}/orders")
    pdf["revenue"] = pdf["price"] * pdf["quantity"]
    res_pd = pdf.groupby("region")["revenue"].sum()

with timer("Spark: ler Parquet + groupBy"):
    res_sp = (
        spark.read.parquet(f"{BASE}/orders")
        .groupBy("region")
        .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
        .collect()
    )

print(f"\nMemória usada pelo DataFrame Pandas: {pdf.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
del pdf

Pandas: ler Parquet + groupby                   0.49 s
Spark: ler Parquet + groupBy                    0.88 s

Memória usada pelo DataFrame Pandas: 660 MB


**Como interpretar:** em uma única máquina, com dados que cabem na RAM, o Pandas frequentemente empata ou ganha. O Spark paga um custo fixo (planejamento, JVM, coordenação de tasks).

O Spark compensa quando:
- os dados **não cabem na memória** de uma máquina (o Pandas simplesmente quebra);
- existe um **cluster**: o mesmo código escala para dezenas de máquinas sem mudanças;
- o pipeline lê de fontes distribuídas (data lake, S3, HDFS).

Por isso, performance em Spark é menos "qual é mais rápido" e mais **"como evitar trabalho desnecessário"**. É isso que o resto da aula aborda.

---
## 3. Como o Spark executa: job, stage e task

Quando você chama uma **ação**, o Spark transforma o plano em trabalho físico:

| Conceito | O que é | Quantos |
|----------|---------|---------|
| **Job** | Trabalho disparado por uma ação (`count`, `write`, `collect`) | 1 por ação (às vezes mais) |
| **Stage** | Bloco de operações que rodam sem trocar dados entre partições | Novo stage a cada shuffle |
| **Task** | Execução de um stage sobre **uma** partição | 1 por partição |

```
  df.filter(...).withColumn(...).groupBy("region").sum()  .write  <- ação
                                                            │
                                                           JOB
            ┌───────────────────────────────────────────────┴────────────┐
         STAGE 1                                                       STAGE 2
   ler + filter + withColumn                                         soma final
   + soma parcial por partição                                       por região
   [T1] [T2] [T3] [T4]   ─────────── SHUFFLE (Exchange) ──────────▶  [T1] [T2] [T3]
```

### Transformações narrow vs wide

| Tipo | Cada partição de saída depende de... | Exemplos | Shuffle? |
|------|-------------------------------------|----------|----------|
| **Narrow** | uma partição de entrada | `select`, `filter`, `withColumn`, `union` | Não |
| **Wide** | várias partições de entrada | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` | Sim |

```
   NARROW                          WIDE (shuffle)
   P1 ──▶ P1'                      P1 ──┬──▶ P1'
   P2 ──▶ P2'                      P2 ──┼──▶ P2'
   P3 ──▶ P3'                      P3 ──┴──▶ P3'
```

O **shuffle** é a operação mais cara do Spark: os dados são serializados, gravados em disco, transferidos pela rede e lidos de novo. **Otimizar Spark é, em grande parte, reduzir shuffles e o volume que passa por eles.**

### 3.1 Lendo o plano com `explain()`

No plano físico, todo shuffle aparece como **`Exchange`**. O plano é lido **de baixo para cima**.

In [6]:
query = (
    orders
    .filter(F.col("price") > 100)                                 # narrow
    .withColumn("revenue", F.col("price") * F.col("quantity"))    # narrow
    .groupBy("region")                                            # wide
    .agg(F.sum("revenue").alias("revenue"))
)

query.explain(mode="formatted")
print("Shuffles no plano:", count_shuffles(query))

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [quantity#21, price#22, region#23]
Batched: true
Location: InMemoryFileIndex [file:/tmp/aula4/orders]
PushedFilters: [IsNotNull(price), GreaterThan(price,100.0)]
ReadSchema: struct<quantity:int,price:double,region:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]

(3) Filter [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]
Condition : (isnotnull(price#22) AND (price#22 > 100.0))

(4) Project [codegen id : 1]
Output [2]: [region#23, (price#22 * cast(quantity#21 as double)) AS revenue#74]
Input [3]: [quantity#21, price#22, region#23]

(5) HashAggregate [codegen id : 1]
Input [2]: [region#23, revenue#74]
Keys [1]: [region#23]
Functions [1]: [partial_sum(revenue#74)]
Aggregate Attributes [1]: [sum#85]
Result

Pontos para observar no plano:

- **`FileScan parquet`** com `PushedFilters`: o filtro `price > 100` foi empurrado para a leitura (*predicate pushdown*).
- **`ReadSchema`**: só as colunas usadas são lidas (*column pruning*). Vantagem do Parquet, que é colunar.
- **`HashAggregate (partial)`** antes do `Exchange`: cada partição soma localmente, e só os totais parciais atravessam o shuffle.
- **`Exchange hashpartitioning(region, 200)`**: o shuffle, com 200 partições de saída.

In [7]:
# Uma query com ordenação + agregação + join: quantos shuffles?
q2 = (
    orders
    .join(products, "product_id")
    .groupBy("category").agg(F.count("*").alias("n"))
    .orderBy(F.desc("n"))
)
print("Shuffles:", count_shuffles(q2))
q2.explain()

Shuffles: 2
== Physical Plan ==
*(4) Sort [n#89L DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(n#89L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=255]
   +- *(3) HashAggregate(keys=[category#30], functions=[count(1)])
      +- Exchange hashpartitioning(category#30, 200), ENSURE_REQUIREMENTS, [plan_id=251]
         +- *(2) HashAggregate(keys=[category#30], functions=[partial_count(1)])
            +- *(2) Project [category#30]
               +- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false
                  :- *(2) Filter isnotnull(product_id#20)
                  :  +- *(2) ColumnarToRow
                  :     +- FileScan parquet [product_id#20] Batched: true, DataFilters: [isnotnull(product_id#20)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<product_id:int>
                  +- BroadcastExchange HashedRelationBroadc

---
## 4. Partições

A partição é a unidade de paralelismo: **uma task processa uma partição**. Poucas partições deixam núcleos ociosos. Partições demais geram overhead de agendamento e muitos arquivos pequenos.

| Momento | Quem decide o número de partições | Configuração |
|---------|----------------------------------|--------------|
| Leitura de arquivos | Tamanho dos arquivos | `spark.sql.files.maxPartitionBytes` (128 MB) |
| Após um shuffle | Configuração fixa (sem AQE) | `spark.sql.shuffle.partitions` (200) |
| Manual | Você | `repartition(n)` / `coalesce(n)` |

Regra prática: partições entre **100 MB e 200 MB**, e pelo menos **2 a 4 vezes o número de núcleos**.

In [8]:
print("Partições na leitura:", orders.rdd.getNumPartitions())

# Quantas linhas há em cada partição?
(
    orders
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show()
)

Partições na leitura: 24
+---------+------+
|partition| count|
+---------+------+
|        0|208334|
|        1|208333|
|        2|208333|
|        3|208333|
|        4|208333|
|        5|208333|
|        6|208333|
|        7|208334|
|        8|208334|
|        9|208333|
|       10|208333|
|       11|208334|
|       12|208333|
|       13|208333|
|       14|208333|
|       15|208333|
|       16|208333|
|       17|208334|
|       18|208334|
|       19|208333|
+---------+------+
only showing top 20 rows


In [9]:
# Efeito de spark.sql.shuffle.partitions em uma agregação
agg = orders.groupBy("customer_id").agg(F.sum("price").alias("total"))

for n in [200, 2, spark.sparkContext.defaultParallelism * 2]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    with timer(f"groupBy com shuffle.partitions = {n}"):
        run(agg)

spark.conf.set("spark.sql.shuffle.partitions", 200)

groupBy com shuffle.partitions = 200            0.83 s
groupBy com shuffle.partitions = 2              0.55 s
groupBy com shuffle.partitions = 48             0.32 s


Com dados pequenos, 200 partições geram 200 tasks minúsculas, e o overhead domina. Com 2, poucas tasks fazem todo o trabalho e os núcleos ficam ociosos. O valor ideal depende do volume, e é exatamente isso que o AQE ajusta sozinho (seção 8).

### `repartition` vs `coalesce`

In [10]:
print("repartition(8):")
orders.repartition(8).explain()

print("coalesce(2):")
orders.coalesce(2).explain()

repartition(8):
== Physical Plan ==
Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=466]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...


coalesce(2):
== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...




| | `repartition(n)` | `coalesce(n)` |
|-|------------------|---------------|
| Shuffle | Sim (`Exchange`) | Não |
| Pode aumentar partições | Sim | Não, só reduz |
| Distribuição | Uniforme | Pode ficar desbalanceada |
| Uso típico | Antes de joins pesados, por chave | Reduzir arquivos antes de escrever |

---
## 5. Estratégias de join

Um join precisa que linhas com a mesma chave estejam na mesma máquina. Há duas formas principais de conseguir isso:

```
  SORT-MERGE JOIN (padrão)                 BROADCAST HASH JOIN
  as duas tabelas passam por shuffle        a tabela pequena é copiada para todos

  orders ──shuffle──┐                       products (pequena)
                    ├── sort + merge          │ cópia │ cópia │ cópia
  customers ─shuffle┘                       [orders P1] [orders P2] [orders P3]
                                            nenhum shuffle da tabela grande
```

| Estratégia | Quando usar | Custo |
|------------|-------------|-------|
| **Broadcast Hash Join** | Um lado é pequeno (padrão: < 10 MB) | Sem shuffle do lado grande; o lado pequeno vai para a memória de cada executor |
| **Sort-Merge Join** | Duas tabelas grandes | Shuffle + ordenação dos dois lados |
| **Shuffle Hash Join** | Um lado médio, sem ordenação | Shuffle dos dois lados, hash do menor |

O Spark escolhe broadcast automaticamente abaixo de `spark.sql.autoBroadcastJoinThreshold`. Vamos desligar esse limite para comparar.

In [11]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # desliga o broadcast automático

smj = orders.join(products, "product_id")
print("Sort-merge — shuffles:", count_shuffles(smj))
with timer("Join orders x products (sort-merge)"):
    run(smj)

bhj = orders.join(F.broadcast(products), "product_id")
print("\nBroadcast  — shuffles:", count_shuffles(bhj))
with timer("Join orders x products (broadcast)"):
    run(bhj)

Sort-merge — shuffles: 2
Join orders x products (sort-merge)             1.47 s

Broadcast  — shuffles: 0
Join orders x products (broadcast)              0.44 s


In [12]:
# Confirme no plano qual estratégia foi usada
print([l.strip() for l in plan_text(smj).splitlines() if "Join" in l])
print([l.strip() for l in plan_text(bhj).splitlines() if "Join" in l])

['+- *(5) SortMergeJoin [product_id#20], [product_id#28], Inner']
['+- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false']


Três formas equivalentes de pedir broadcast:

```python
orders.join(F.broadcast(products), "product_id")      # função
orders.join(products.hint("broadcast"), "product_id") # hint na API
spark.sql("SELECT /*+ BROADCAST(p) */ * FROM orders o JOIN products p USING (product_id)")
```

> Cuidado: fazer broadcast de uma tabela grande pode estourar a memória do driver e dos executores. Use para dimensões pequenas.

In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10MB")   # restaura o padrão

---
## 6. Data skew (dados desbalanceados)

Skew acontece quando poucas chaves concentram grande parte das linhas. Depois do shuffle, **uma partição fica enorme** e sua task demora muito mais que as outras. O job inteiro espera por ela.

```
   Tasks após o shuffle (tempo)
   T1 ███
   T2 ███
   T3 ████████████████████████████████   <- chave "quente"
   T4 ███
```

Sintoma típico na Spark UI: na aba Stages, a duração máxima das tasks é muito maior que a mediana, e o job fica parado em "199/200 tasks".

Vamos simular: **40% dos pedidos** pertencem ao `customer_id = 0` (por exemplo, um marketplace que compra em nome de terceiros).

In [14]:
orders_skew = orders.withColumn(
    "customer_id",
    F.when(F.rand(7) < 0.4, F.lit(0)).otherwise(F.col("customer_id"))
)
orders_skew.write.mode("overwrite").parquet(f"{BASE}/orders_skew")
orders_skew = spark.read.parquet(f"{BASE}/orders_skew")

# 1) Detectar: quais chaves concentram mais linhas?
(
    orders_skew.groupBy("customer_id").count()
    .withColumn("pct", F.round(F.col("count") / N_ORDERS * 100, 2))
    .orderBy(F.desc("count"))
    .show(5)
)

+-----------+-------+----+
|customer_id|  count| pct|
+-----------+-------+----+
|          0|1999763|40.0|
|      32361|     35| 0.0|
|     103645|     35| 0.0|
|     100499|     35| 0.0|
|      72640|     34| 0.0|
+-----------+-------+----+
only showing top 5 rows


In [15]:
# 2) Ver o efeito nas partições depois de um shuffle pela chave
(
    orders_skew.repartition(8, "customer_id")
    .groupBy(F.spark_partition_id().alias("partition")).count()
    .orderBy(F.desc("count"))
    .show()
)

+---------+-------+
|partition|  count|
+---------+-------+
|        7|2374060|
|        2| 378545|
|        3| 377244|
|        6| 375019|
|        4| 374651|
|        5| 374332|
|        1| 373141|
|        0| 373008|
+---------+-------+



In [16]:
# 3) Uma métrica simples: maior partição vs mediana
partition_balance(orders, "customer_id", label="orders")
partition_balance(orders_skew, "customer_id", label="orders_skew")

orders       maior =   317,729 | mediana =   311,943 | razão = 1.0x
orders_skew  maior = 2,186,568 | mediana =   187,215 | razão = 11.7x


Em um cluster com 100 núcleos, o stage dura o tempo da **maior** task. Com uma partição 10x maior que a mediana, 99 núcleos ficam esperando por um.

> Em modo local, com 2 a 8 núcleos, o tempo total quase não muda: o trabalho total é o mesmo e há poucos núcleos para ficarem ociosos. Por isso, aqui medimos o skew pela distribuição das partições, que é o que a aba Stages mostra em produção.

### 6.1 Correções

| Técnica | Ideia | Quando |
|---------|-------|--------|
| **Broadcast** | Elimina o shuffle, então o skew deixa de importar | O outro lado cabe em memória |
| **Salting** | Divide a chave quente em N subchaves artificiais | Os dois lados são grandes |
| **AQE skew join** | O Spark detecta partições enormes e as divide sozinho | Spark 3+; ver seção 8 |
| **Tratar a chave à parte** | Processa a chave quente separadamente e faz `union` | Poucas chaves conhecidas |

**Salting**: adicionamos um número aleatório (`salt`) de 0 a N-1 do lado grande, e replicamos cada linha do lado pequeno N vezes, uma por salt. O join passa a ser por `(customer_id, salt)`, e a chave quente se espalha em N partições.

```
  orders_skew                     customers (replicada N vezes)
  cust=0, salt=3   ─────────▶     cust=0, salt=0..N-1
  cust=0, salt=7                  cust=1, salt=0..N-1
  cust=0, salt=1                  ...
```

In [17]:
SALT = 16

orders_salted = orders_skew.withColumn("salt", (F.rand(8) * SALT).cast("int"))
customers_salted = customers.crossJoin(
    spark.range(SALT).select(F.col("id").cast("int").alias("salt"))
)

partition_balance(orders_skew, "customer_id", label="sem salt")
partition_balance(orders_salted, "customer_id", "salt", label="com salt")

# O resultado do join continua correto: mesmo número de linhas
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
n_plain  = orders_skew.join(customers, "customer_id").count()
n_salted = orders_salted.join(customers_salted, ["customer_id", "salt"]).count()
print(f"\nlinhas sem salt: {n_plain:,} | com salt: {n_salted:,}")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

sem salt     maior = 2,186,568 | mediana =   187,215 | razão = 11.7x
com salt     maior =   441,645 | mediana =   308,298 | razão = 1.4x

linhas sem salt: 5,000,000 | com salt: 5,000,000


O salting tem custo: o lado pequeno fica N vezes maior e há mais dados no shuffle. Se a dimensão cabe em memória, **broadcast é a solução mais simples**, porque não há shuffle e o skew deixa de existir.

---
## 7. Cache e persist

Por causa da avaliação preguiçosa, **cada ação recalcula o plano inteiro desde a leitura**. Se um DataFrame caro é usado várias vezes, o cache evita esse retrabalho.

| Nível (`StorageLevel`) | Onde guarda | Observação |
|------------------------|-------------|------------|
| `MEMORY_AND_DISK` | Memória, transbordando para disco | Padrão de `df.cache()` |
| `MEMORY_ONLY` | Só memória | O que não cabe é recalculado |
| `DISK_ONLY` | Só disco | Útil quando o cálculo é muito caro |

**Use cache quando:** o mesmo DataFrame alimenta várias ações (relatórios, loops, algoritmos iterativos de ML).

**Evite quando:** o DataFrame é usado uma única vez, ou é só uma leitura simples de Parquet (ler de novo é barato). O cache ocupa memória que os shuffles poderiam usar.

In [18]:
spark.conf.set("spark.sql.shuffle.partitions", spark.sparkContext.defaultParallelism * 2)

# Base cara de calcular, mas pequena (uma linha por cliente): join + agregação com countDistinct
customer_features = (
    orders.join(products, "product_id")
    .groupBy("customer_id")
    .agg(
        F.sum(F.col("price") * F.col("quantity")).alias("revenue"),
        F.count("*").alias("n_orders"),
        F.countDistinct("category").alias("n_categories"),
    )
    .join(customers, "customer_id")
)

# Ex.: exportar um arquivo por segmento. Cada exportação é uma ação separada.
def export_by_segment(df):
    for seg in ["Consumer", "Corporate", "Small Business"]:
        run(df.filter(F.col("segment") == seg))

with timer("3 exportações SEM cache"):
    export_by_segment(customer_features)

features_cached = customer_features.cache()
with timer("Materializar o cache (count)"):
    features_cached.count()

with timer("3 exportações COM cache"):
    export_by_segment(features_cached)

_ = features_cached.unpersist()

3 exportações SEM cache                         2.91 s
Materializar o cache (count)                    0.86 s
3 exportações COM cache                         0.22 s


`cache()` também é preguiçoso: o DataFrame só é guardado na primeira ação. Por isso materializamos com `count()`. Na Spark UI, a aba **Storage** mostra o que está em cache e quanto ocupa.

O cache compensou porque a base é **cara de calcular e pequena para guardar** (200 mil linhas). No exercício 4 você verá o caso oposto.

---
## 8. Adaptive Query Execution (AQE)

Sem AQE, o plano é decidido **antes** da execução, com estimativas. Com AQE (padrão desde o Spark 3.2), o Spark **reotimiza entre os stages**, usando as estatísticas reais de cada shuffle.

| Recurso do AQE | O que faz | Configuração |
|----------------|-----------|--------------|
| Coalescer partições | Junta partições pequenas após o shuffle (200 vira poucas) | `spark.sql.adaptive.coalescePartitions.enabled` |
| Trocar estratégia de join | Converte sort-merge em broadcast se um lado ficou pequeno após filtros | `spark.sql.adaptive.autoBroadcastJoinThreshold` |
| Skew join | Divide partições gigantes em pedaços menores | `spark.sql.adaptive.skewJoin.enabled` |

Até agora o AQE estava desligado. Vamos ligá-lo.

In [19]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
agg = orders.groupBy("region").agg(F.sum("price").alias("total"))

spark.conf.set("spark.sql.adaptive.enabled", "false")
print("Sem AQE, partições após o shuffle:", agg.rdd.getNumPartitions())

spark.conf.set("spark.sql.adaptive.enabled", "true")
agg = orders.groupBy("region").agg(F.sum("price").alias("total"))
print("Com AQE, partições após o shuffle:", agg.rdd.getNumPartitions())

Sem AQE, partições após o shuffle: 200
Com AQE, partições após o shuffle: 1


In [20]:
# AQE skew join: reduzimos os limiares para que o nosso dataset de exemplo seja considerado "skewed"
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256KB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "256KB")

skew_join = (
    orders_skew.join(customers, "customer_id")
    .groupBy("segment").agg(F.count("*").alias("orders"))
)
print("Antes de executar:")
print([l.strip() for l in plan_text(skew_join).splitlines() if "AdaptiveSparkPlan" in l])

skew_join.collect()   # ação no próprio DataFrame, para podermos ver o plano final dele

print("\nDepois de executar:")
final_plan = plan_text(skew_join)
print([l.strip() for l in final_plan.splitlines()
       if "AdaptiveSparkPlan" in l or "SortMergeJoin" in l or "AQEShuffleRead" in l][:4])

Antes de executar:
['AdaptiveSparkPlan isFinalPlan=false']

Depois de executar:
['AdaptiveSparkPlan isFinalPlan=true', '+- AQEShuffleRead coalesced', '+- *(5) SortMergeJoin [customer_id#138], [customer_id#25], Inner', ':  +- AQEShuffleRead coalesced']


Antes da execução, o plano mostra `isFinalPlan=false`. Depois, mostra `isFinalPlan=true` e `SortMergeJoin(skew=true)`: o AQE detectou a partição quente e a dividiu. `AQEShuffleRead coalesced and skewed` indica partições juntadas e divididas.

> Em produção, mantenha o AQE ligado (padrão). Ele não substitui um bom desenho, mas resolve boa parte dos ajustes de partição automaticamente.

In [21]:
# Restaurar configurações padrão
for k in ["spark.sql.autoBroadcastJoinThreshold",
          "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    spark.conf.unset(k)

---
## 9. UDF Python vs funções nativas vs Pandas UDF

Funções nativas (`F.*`) rodam dentro da JVM e são otimizadas pelo Catalyst. Uma UDF Python obriga o Spark a **serializar cada linha, enviá-la a um processo Python e trazer o resultado de volta**.

```
  F.when(...)     JVM ──────────────────────────────▶ JVM            (rápido)
  @udf            JVM ─▶ serializa linha a linha ─▶ Python ─▶ JVM     (lento)
  @pandas_udf     JVM ─▶ lotes Arrow (colunar) ───▶ Pandas ─▶ JVM     (intermediário)
```

In [22]:
from pyspark.sql.functions import pandas_udf

def classify(price):
    if price is None: return None
    if price < 100:   return "Budget"
    if price < 300:   return "Mid-range"
    return "Premium"

classify_udf = F.udf(classify, "string")

@pandas_udf("string")
def classify_pandas(price: pd.Series) -> pd.Series:
    return pd.Series(np.select([price < 100, price < 300], ["Budget", "Mid-range"], default="Premium"))

native = (F.when(F.col("price") < 100, "Budget")
           .when(F.col("price") < 300, "Mid-range")
           .otherwise("Premium"))

with timer("Função nativa (F.when)"):
    run(orders.withColumn("cls", native))
with timer("Pandas UDF (Arrow)"):
    run(orders.withColumn("cls", classify_pandas("price")))
with timer("UDF Python linha a linha"):
    run(orders.withColumn("cls", classify_udf("price")))

Função nativa (F.when)                          0.43 s
Pandas UDF (Arrow)                              1.76 s
UDF Python linha a linha                        0.79 s


Ordem de preferência: **função nativa > Pandas UDF > UDF Python**. Antes de escrever uma UDF, procure na [lista de funções](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): quase sempre já existe uma.

---
## 10. Spark UI: onde olhar

Abra a URL impressa no setup enquanto as células rodam. (No Colab a UI não fica acessível diretamente; rode localmente para explorá-la.)

| Aba | O que procurar |
|-----|----------------|
| **Jobs** | Quais ações demoraram mais (os nomes vêm do `timer`) |
| **Stages** | Shuffle Read/Write, e duração **máxima vs mediana** das tasks (skew) |
| **Storage** | DataFrames em cache e quanto ocupam |
| **SQL / DataFrame** | Plano visual com linhas e tempo por operador |
| **Executors** | Memória, tempo de GC, tasks com falha |

---
## 11. Exercícios

> Use `count_shuffles`, `explain()` e `timer` para justificar cada resposta.

In [23]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# Quantos shuffles tem a query abaixo? Explique de onde vem cada um.
# Depois, reescreva-a para eliminar pelo menos um shuffle sem mudar o resultado.

ex1 = (
    orders
    .join(customers, "customer_id")
    .join(products, "product_id")
    .groupBy("segment", "category")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
    .orderBy(F.desc("revenue"))
)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # simula dimensões "grandes"
# Sua solução:

In [24]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# Em orders_skew, descubra:
#   a) qual percentual das linhas a chave mais frequente representa;
#   b) a razão entre a maior e a mediana das partições após repartition(16, "customer_id").
# Uma razão acima de ~5 é um bom indício de skew.

# Sua solução:

In [25]:
# ── Exercício 3 ─────────────────────────────────────────────────────────────
# A função abaixo usa uma UDF para calcular o trimestre ("Q1".."Q4") de order_date.
# Reescreva usando apenas funções nativas e compare os tempos.

quarter_udf = F.udf(lambda d: f"Q{(d.month - 1) // 3 + 1}", "string")

# Sua solução:

In [26]:
# ── Exercício 4 ─────────────────────────────────────────────────────────────
# Você precisa gerar 4 relatórios a partir de:
#   orders filtrado em price > 200, com join em customers, e coluna revenue.
# Meça o tempo total com e sem cache. Em qual cenário o cache NÃO valeria a pena?

# Sua solução:

### Soluções

In [27]:
# ── Solução 1 ───────────────────────────────────────────────────────────────
print("Original — shuffles:", count_shuffles(ex1))
# 2 joins sort-merge = 4 Exchanges (cada lado de cada join),
# + 1 do groupBy, + 1 do orderBy (range partitioning) = 6.
# (O Spark pode reaproveitar o particionamento e mostrar um pouco menos.)

ex1_opt = (
    orders
    .join(F.broadcast(customers), "customer_id")
    .join(F.broadcast(products), "product_id")
    .groupBy("segment", "category")
    .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
print("Otimizada — shuffles:", count_shuffles(ex1_opt))

with timer("Original"):
    run(ex1)
with timer("Com broadcast"):
    run(ex1_opt)

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

Original — shuffles: 6
Otimizada — shuffles: 2
Original                                        1.65 s
Com broadcast                                   0.55 s


In [28]:
# ── Solução 2 ───────────────────────────────────────────────────────────────
top = orders_skew.groupBy("customer_id").count().orderBy(F.desc("count")).first()
print(f"a) customer_id={top['customer_id']} concentra {top['count'] / N_ORDERS:.1%} das linhas")

sizes = (
    orders_skew.repartition(16, "customer_id")
    .groupBy(F.spark_partition_id()).count()
    .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
    .first()
)
print(f"b) maior = {sizes['max']:,} | mediana = {sizes['median']:,} | razão = {sizes['max'] / sizes['median']:.1f}x")

a) customer_id=0 concentra 40.0% das linhas
b) maior = 2,186,568 | mediana = 187,215 | razão = 11.7x


In [29]:
# ── Solução 3 ───────────────────────────────────────────────────────────────
quarter_native = F.concat(F.lit("Q"), F.quarter("order_date"))

with timer("UDF"):
    run(orders.withColumn("q", quarter_udf("order_date")))
with timer("Nativa (F.quarter)"):
    run(orders.withColumn("q", quarter_native))

orders.select("order_date", quarter_native.alias("q")).show(3)

UDF                                             1.77 s
Nativa (F.quarter)                              0.47 s
+----------+---+
|order_date|  q|
+----------+---+
|2024-12-19| Q4|
|2024-01-28| Q1|
|2024-01-20| Q1|
+----------+---+
only showing top 3 rows


In [30]:
# ── Solução 4 ───────────────────────────────────────────────────────────────
base = (
    orders.filter(F.col("price") > 200)
    .join(customers, "customer_id")
    .withColumn("revenue", F.col("price") * F.col("quantity"))
)

def four_reports(df):
    run(df.groupBy("region").agg(F.sum("revenue")))
    run(df.groupBy("segment").agg(F.sum("revenue")))
    run(df.groupBy("order_date").agg(F.count("*")))
    run(df.groupBy("signup_year").agg(F.avg("price")))

with timer("4 relatórios sem cache"):
    four_reports(base)

with timer("4 relatórios com cache (inclui materializar)"):
    cached = base.cache()
    cached.count()
    four_reports(cached)
_ = cached.unpersist()

# Aqui o cache tende a NÃO compensar:
# - a base é larga (milhões de linhas, todas as colunas) e cara de guardar;
# - cada relatório usa poucas colunas, e sem cache o Spark lê só essas (column pruning);
# - recalcular (ler Parquet + broadcast join) é barato.
# Compare com a seção 7: lá a base era pequena e cara de calcular.

4 relatórios sem cache                          1.32 s
4 relatórios com cache (inclui materializar)    1.87 s


---
## Resumo: sintoma, causa e ação

| Sintoma | Causa provável | O que fazer |
|---------|----------------|-------------|
| Muitos `Exchange` no plano | Joins e agregações em excesso | Broadcast de dimensões, agregar antes de juntar |
| Job parado em "199/200 tasks" | Data skew | Broadcast, salting, AQE skew join |
| Milhares de tasks de milissegundos | Partições demais | AQE ligado, ajustar `shuffle.partitions` |
| Poucos núcleos ocupados | Partições de menos | `repartition`, ler arquivos menores |
| Mesmo cálculo repetido | Reuso sem cache | `cache()` + `unpersist()` |
| Coluna calculada muito lenta | UDF Python | Função nativa ou Pandas UDF |
| Leitura lenta | CSV/JSON, sem pruning | Parquet/Delta, filtrar e selecionar cedo |

**Próxima aula:** pipeline ETL completo com arquitetura Medallion e Delta Lake.

In [31]:
# spark.stop()